# Predicción de Precios de Viviendas usando Random Forest (scikit-learn)

Este notebook te guía para entrenar un modelo Random Forest usando scikit-learn en el dataset de Precios de Viviendas.

El código se verá aproximadamente así:

```python
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

dataset = pd.read_csv("project/dataset.csv")
X = dataset.drop('SalePrice', axis=1)
y = dataset['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print(model.score(X_test, y_test))
```

Random Forest (Bosque de Decisión) es una familia de modelos basados en árboles que incluye Random Forests y Gradient Boosted Trees. Son el mejor punto de salida cuando trabajas con datos tabulares.

## Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import LabelEncoder

# Comenta esto si las visualizaciones no funcionan en tu lado
%matplotlib inline

## Cargar el dataset

In [ ]:
train_file_path = "train.csv"
dataset_df = pd.read_csv(train_file_path)
print("Full train dataset shape is {}".format(dataset_df.shape))

Los datos están compuestos por 81 columnas y 1460 registros. Podemos ver las 81 dimensiones de nuestro dataset imprimiendo las primeras 3 entradas usando el siguiente código:

In [ ]:
dataset_df.head(3)

- Hay 79 columnas de características. Usando estas características, tu modelo tiene que predecir el precio de venta de la vivienda indicado por la columna etiqueta llamada `SalePrice`.

Eliminaremos la columna `Id` ya que no es necesaria para el entrenamiento del modelo.

In [ ]:
dataset_df = dataset_df.drop('Id', axis=1)
dataset_df.head(3)

Podemos inspeccionar los tipos de las columnas de características usando el siguiente código:

In [ ]:
dataset_df.info()

## Manejo de valores faltantes

In [ ]:
# Ver valores faltantes
missing_values = dataset_df.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
print("Columnas con valores faltantes:")
print(missing_values)

Vamos a manejar los valores faltantes:

In [ ]:
# Separar características y objetivo
X = dataset_df.drop('SalePrice', axis=1)
y = dataset_df['SalePrice']

# Identificar columnas numéricas y categóricas
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f"Columnas numéricas: {len(numeric_cols)}")
print(f"Columnas categóricas: {len(categorical_cols)}")

In [ ]:
# Rellenar valores faltantes
# Para numéricos: usar la mediana
for col in numeric_cols:
    X[col] = X[col].fillna(X[col].median())

# Para categóricos: usar la moda
for col in categorical_cols:
    X[col] = X[col].fillna(X[col].mode()[0])

# Verificar que no hay valores faltantes
print(f"Valores faltantes restantes: {X.isnull().sum().sum()}")

## Codificar variables categóricas

In [ ]:
# Usar Label Encoding para variables categóricas
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

print(f"Dataset shape after encoding: {X.shape}")
X.head()

## División de datos

In [ ]:
# Dividir los datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

print("{} examples in training, {} examples in testing.".format(
    len(X_train), len(X_test)))

## Distribución de Precios de Viviendas

Ahora echemos un vistazo a cómo se distribuyen los precios de las viviendas.

In [ ]:
print(y.describe())
plt.figure(figsize=(9, 8))
sns.histplot(y, color='g', bins=50, alpha=0.4)
plt.title('Distribución de Precios de Viviendas')
plt.xlabel('SalePrice')
plt.ylabel('Frecuencia')
plt.show()

## Entrenar el modelo Random Forest

In [ ]:
# Crear y entrenar el modelo
rf_model = RandomForestRegressor(
_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

print("Modelo entrenado exitosamente!")

## Predicciones y evaluación

In [ ]:
# Predicciones
y_pred = rf_model.predict(X_test)

# Métricas de evaluación
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("=== Métricas de Evaluación ===")
print(f"MSE: {mse:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"MAE: {mae:,.2f}")
print(f"R2 Score: {r2:.4f}")

## Validación Cruzada

In [ ]:
# Validación cruzada
cv_scores = cross_val_score(rf_model, X, y, cv=5, scoring='r2')

print("=== Validación Cruzada (R2) ===")
print(f"Scores por fold: {cv_scores}")
print(f"Media: {cv_scores.mean():.4f}")
print(f"Desviación estándar: {cv_scores.std():.4f}")

## Importancia de variables

In [ ]:
# Importancia de características
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

# Top 15 características
plt.figure(figsize=(12, 8))
top_features = feature_importance.head(15)
plt.barh(range(len(top_features)), top_features['importance'], align='center')
plt.yticks(range(len(top_features)), top_features['feature'])
plt.gca().invert_yaxis()
plt.xlabel('Importancia')
plt.title('Top 15 Características más Importantes')
plt.tight_layout()
plt.show()

print("\nTop 10 características:")
print(feature_importance.head(10))

## Comparar con Gradient Boosting

In [ ]:
# Entrenar Gradient Boosting
gb_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb_model.fit(X_train, y_train)

# Predicciones
y_pred_gb = gb_model.predict(X_test)

# Métricas
r2_gb = r2_score(y_test, y_pred_gb)
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))

print("=== Gradient Boosting ===")
print(f"R2 Score: {r2_gb:.4f}")
print(f"RMSE: {rmse_gb:,.2f}")

print("\n=== Random Forest ===")
print(f"R2 Score: {r2:.4f}")
print(f"RMSE: {rmse:,.2f}")

## Predicciones en datos de prueba

In [ ]:
# Cargar datos de prueba
test_file_path = "test.csv"
test_data = pd.read_csv(test_file_path)
ids = test_data['Id']
test_data = test_data.drop('Id', axis=1)

# Aplicar el mismo preprocesamiento
# Rellenar valores faltantes
for col in numeric_cols:
    if col in test_data.columns:
        test_data[col] = test_data[col].fillna(X[col].median())

for col in categorical_cols:
    if col in test_data.columns:
        test_data[col] = test_data[col].fillna(X[col].mode()[0])

# Codificar variables categóricas
for col in categorical_cols:
    if col in test_data.columns:
        le = label_encoders[col]
        # Manejar valores unseen
        test_data[col] = test_data[col].apply(
            lambda x: le.transform([str(x)])[0] if str(x) in le.classes_ else -1
        )

print(f"Test data shape: {test_data.shape}")

In [ ]:
# Asegurar que las columnas coincidan
missing_cols = set(X.columns) - set(test_data.columns)
for col in missing_cols:
    test_data[col] = 0

test_data = test_data[X.columns]

# Predicciones
preds = rf_model.predict(test_data)

output = pd.DataFrame({'Id': ids, 'SalePrice': preds})
output.head()

In [ ]:
# Guardar submission
sample_submission = pd.read_csv('sample_submission.csv')
sample_submission['SalePrice'] = preds
sample_submission.to_csv('submission.csv', index=False)
sample_submission.head()